# 🏥 Healthcare Claims AI Agent
### TP01 — Optimized | LangChain · LangGraph · Google Gemini 2.5 Flash

---

## Overview

This notebook implements a **conversational AI agent** for healthcare claims analytics.
It ingests structured Excel claims data, detects billing anomalies, and answers
natural language questions through a **LangChain ReAct agent** backed by Google Gemini.

### Capabilities
- **Anomaly Detection** — identifies the most significant month-over-month dollar swing in billed amounts and pinpoints its regional and claim-type drivers
- **MoM Driver Decomposition** — breaks down paid claims changes into percentage-point contributions by Region × Claim Type
- **Conversational Interface** — an interactive chat UI lets analysts query findings in plain English without writing any code

### Architecture
```
User uploads Excel file
         │
         ▼
LangGraph ReAct Agent  ◄──  Google Gemini 2.5 Flash
         │
         ├──► Tool 1: investigate_claims_spike()
         │         Detects largest MoM $ change in BILLED amounts
         │         Renders annotated stacked bar chart
         │
         └──► Tool 2: analyze_incremental_paid_claims()
                   Decomposes MoM % change in PAID amounts
                   by Region × Claim Type (percentage points)
                   Renders faceted bar chart
```

### Data Requirements
Excel file with two sheets:
- `fake enrollment` — Member_ID, Region
- `fake claims` — Member_ID, Service_Date, Billed_Amt, Paid_Amt, Type


In [ ]:
# Cell 1 — Install Dependencies
# Run once. Restart runtime if prompted.

import subprocess
subprocess.run([
    "pip", "install", "-q", "-U",
    "langchain", "langchain-google-genai", "langchain-core",
    "langchain-community", "langgraph",
    "plotly", "ipywidgets", "pandas", "xlrd", "openpyxl"
], check=True)

from google.colab import output as colab_output
colab_output.enable_custom_widget_manager()

print("✅ Dependencies installed and widget rendering enabled")


In [ ]:
%%writefile claims_tools.py
# Cell 2 — Define AI Tools
# Writes claims_tools.py to disk so the agent can import it.

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display
from langchain.tools import tool


def _parse_dates(df, date_col='Service_Date'):
    """Robustly parse date columns from both numeric (Excel serial) and string formats."""
    if pd.api.types.is_numeric_dtype(df[date_col]):
        df['Date'] = pd.to_datetime(df[date_col], unit='D', origin='1899-12-30')
    else:
        df['Date'] = pd.to_datetime(df[date_col], format='%m/%d/%Y', errors='coerce')
        df['Date'] = df['Date'].fillna(pd.to_datetime(df[date_col], errors='coerce'))
    return df


def _load_claims(file_path):
    """Load and merge claims + enrollment sheets from Excel."""
    enrollment = pd.read_excel(file_path, sheet_name='fake enrollment')
    claims     = pd.read_excel(file_path, sheet_name='fake claims')
    return pd.merge(claims, enrollment, on='Member_ID', how='inner')


# ---------------------------------------------------------
# TOOL 1: Stacked Bar Anomaly Investigator
# ---------------------------------------------------------

@tool
def investigate_claims_spike(file_path: str) -> str:
    """
    Analyzes hospital claims data from an Excel file to find the most significant
    month-over-month dollar change in BILLED amounts, identifies its regional and
    claim-type drivers, and renders an annotated stacked bar chart.

    Args:
        file_path: Path to the Excel claims file.

    Returns:
        Plain-text summary of the anomaly month, swing magnitude, and top driver.
    """
    try:
        df = _load_claims(file_path)
    except Exception as e:
        return f"Error loading file: {e}"

    df = _parse_dates(df)
    df['YearMonth'] = df['Date'].dt.to_period('M')

    monthly = (
        df.groupby('YearMonth')['Billed_Amt']
        .sum().reset_index().sort_values('YearMonth')
    )
    if len(monthly) < 2:
        return "Insufficient data: at least two months of claims are required."

    monthly['Dollar_Change'] = monthly['Billed_Amt'].diff()
    idx          = monthly['Dollar_Change'].abs().idxmax()
    spike_month  = monthly.loc[idx, 'YearMonth']
    spike_amt    = monthly.loc[idx, 'Billed_Amt']
    dollar_swing = monthly.loc[idx, 'Dollar_Change']
    prev_amt     = monthly.loc[idx - 1, 'Billed_Amt']
    spike_pct    = (dollar_swing / prev_amt * 100) if prev_amt else 0
    direction    = "growth" if dollar_swing > 0 else "decline"
    sign         = "+" if dollar_swing > 0 else ""

    drivers = (
        df[df['YearMonth'] == spike_month]
        .groupby(['Type', 'Region'])['Billed_Amt']
        .sum().reset_index().sort_values('Billed_Amt', ascending=False)
    )
    top = drivers.iloc[0]

    stacked = (
        df.groupby(['YearMonth', 'Region'])['Billed_Amt']
        .sum().reset_index().sort_values('YearMonth')
    )
    stacked['Plot_Date'] = stacked['YearMonth'].dt.to_timestamp()
    spike_month_ts       = spike_month.to_timestamp()

    fig = px.bar(
        stacked, x='Plot_Date', y='Billed_Amt', color='Region', barmode='stack',
        title="Monthly Billed Amount — Anomaly Detection",
        labels={'Billed_Amt': 'Total Change in Billed ($)', 'Plot_Date': 'Month'}
    )
    fig.update_xaxes(dtick="M1", tickformat="%b %Y", tickangle=-30)
    fig.add_trace(go.Scatter(
        x=[spike_month_ts], y=[spike_amt], mode='markers',
        marker=dict(size=35, symbol='circle-open', color='red', line=dict(width=4)),
        showlegend=False, hoverinfo='skip'
    ))
    fig.add_annotation(
        x=spike_month_ts, y=spike_amt,
        text=(f"Anomaly: {sign}${abs(dollar_swing):,.0f} ({sign}{spike_pct:.1f}%)<br>"
              f"Driver: {top['Type']} — {top['Region']}"),
        showarrow=True, arrowhead=2, arrowcolor="red",
        font=dict(color="red", size=12), ay=-50
    )
    display(go.FigureWidget(fig))

    return (
        f"The most significant change occurred in {spike_month} with a swing of "
        f"${abs(dollar_swing):,.2f} ({abs(spike_pct):.1f}% {direction}). "
        f"Primary driver: '{top['Type']}' claims in '{top['Region']}' "
        f"(${top['Billed_Amt']:,.2f})."
    )


# ---------------------------------------------------------
# TOOL 2: MoM Percentage Point Driver Decomposition
# ---------------------------------------------------------

@tool
def analyze_incremental_paid_claims(file_path: str) -> str:
    """
    Analyzes PAID claims month-over-month. Decomposes Region x Claim Type drivers
    as percentage-point contributions to the total MoM change, and renders a
    faceted bar chart.

    Args:
        file_path: Path to the Excel claims file.

    Returns:
        Plain-text summary of the most recent MoM change and its top driver.
    """
    try:
        df = _load_claims(file_path)
    except Exception as e:
        return f"Error loading file: {e}"

    df = _parse_dates(df)
    df['YearMonth'] = df['Date'].dt.to_period('M').astype(str)
    unique_months   = sorted(df['YearMonth'].unique())

    if len(unique_months) < 2:
        return "Insufficient data: at least two months of claims are required."

    monthly_totals = df.groupby('YearMonth')['Paid_Amt'].sum()
    pivot_df       = df.pivot_table(
        index='YearMonth', columns=['Region', 'Type'],
        values='Paid_Amt', aggfunc='sum', fill_value=0
    )
    pct_pt_df = (pivot_df.diff().div(monthly_totals.shift(1), axis=0) * 100).dropna()

    plot_df              = pct_pt_df.stack(['Region', 'Type']).reset_index(name='Pct_Pt_Contribution')
    plot_df['Plot_Date'] = pd.to_datetime(plot_df['YearMonth'])

    fig = px.bar(
        plot_df, x='Plot_Date', y='Pct_Pt_Contribution',
        color='Region', facet_col='Type', barmode='relative',
        title="MoM Paid Claims — Percentage Point Driver Decomposition",
        labels={'Pct_Pt_Contribution': 'Contribution to MoM Change (pp)', 'Plot_Date': 'Month'}
    )
    fig.update_xaxes(dtick="M1", tickformat="%b %Y", tickangle=-30)
    fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
    display(go.FigureWidget(fig))

    target_month = pct_pt_df.index[-1]
    prev_month   = unique_months[unique_months.index(target_month) - 1]
    total_curr   = monthly_totals[target_month]
    total_prev   = monthly_totals[prev_month]
    mom_pct      = ((total_curr - total_prev) / total_prev * 100) if total_prev else 0

    top = (
        plot_df[plot_df['YearMonth'] == target_month]
        .assign(Abs=lambda x: x['Pct_Pt_Contribution'].abs())
        .sort_values('Abs', ascending=False)
        .iloc[0]
    )
    direction = "increase" if mom_pct > 0 else "decrease"

    return (
        f"In {target_month}, total paid claims were ${total_curr:,.2f} — "
        f"a {abs(mom_pct):.1f}% {direction} from {prev_month}. "
        f"Primary driver: '{top['Type']}' claims in '{top['Region']}' "
        f"({top['Pct_Pt_Contribution']:+.1f} pp contribution)."
    )

print("✅ claims_tools.py written to disk")


In [ ]:
# Cell 3 — Build the ReAct Agent & Launch Interactive UI

import os
import importlib
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output
from google.colab import files, userdata

import claims_tools
importlib.reload(claims_tools)
from claims_tools import investigate_claims_spike, analyze_incremental_paid_claims

from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.prebuilt import create_react_agent

# ── Initialise LLM & agent ──────────────────────────────────
os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
llm   = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)
agent = create_react_agent(model=llm, tools=[investigate_claims_spike, analyze_incremental_paid_claims])

# ── State ────────────────────────────────────────────────────
uploaded_file_path = None

# ── Widgets ──────────────────────────────────────────────────
upload_btn = widgets.Button(
    description='📂 Upload Claims File',
    button_style='info',
    layout=widgets.Layout(width='220px', height='36px')
)
file_label = widgets.HTML(value="<i style='color:#888'>No file uploaded yet</i>")
plot_btn   = widgets.Button(
    description='📊 Plot Baseline Trend',
    button_style='primary',
    layout=widgets.Layout(width='200px', height='36px'),
    disabled=True
)
chat_input = widgets.Text(
    placeholder='Ask the agent a question about your claims data…',
    layout=widgets.Layout(width='68%')
)
chat_btn  = widgets.Button(description='Send', button_style='success', icon='paper-plane')
chat_box  = widgets.HBox([widgets.Label("💬"), chat_input, chat_btn])
chat_box.layout.display = 'none'
chart_area = widgets.Output()
chat_area  = widgets.Output(layout=widgets.Layout(
    height='300px', overflow_y='auto',
    border='1px solid #e0e0e0', padding='12px', background_color='#fafafa'
))

# ── Upload ───────────────────────────────────────────────────
def on_upload_click(b):
    global uploaded_file_path
    with chat_area:
        clear_output(wait=True)
        print("📂 Opening file picker — select your Excel claims file (.xls or .xlsx)…")

    uploaded = files.upload()

    if not uploaded:
        with chat_area:
            clear_output(wait=True)
            print("⚠️  No file selected. Please try again.")
        return

    filename = list(uploaded.keys())[0]
    if not filename.lower().endswith(('.xls', '.xlsx')):
        with chat_area:
            clear_output(wait=True)
            print(f"❌  '{filename}' is not an Excel file. Please upload a .xls or .xlsx file.")
        return

    uploaded_file_path = f"/content/{filename}"
    with open(uploaded_file_path, 'wb') as f:
        f.write(uploaded[filename])

    file_label.value  = f"<b style='color:#2a6b2a'>✅ Loaded: {filename}</b>"
    plot_btn.disabled = False

    with chat_area:
        clear_output(wait=True)
        print(f"✅ File uploaded: {filename}")
        print("─" * 60)
        print("Click '📊 Plot Baseline Trend' to visualise your data,")
        print("then use the chat to ask the agent questions.")
        print("─" * 60)

# ── Plot ─────────────────────────────────────────────────────
def on_plot_click(b):
    if not uploaded_file_path:
        with chat_area:
            print("⚠️  Please upload a file first.")
        return

    with chart_area:
        clear_output(wait=True)
        try:
            df = pd.merge(
                pd.read_excel(uploaded_file_path, sheet_name='fake claims'),
                pd.read_excel(uploaded_file_path, sheet_name='fake enrollment'),
                on='Member_ID', how='inner'
            )
            if pd.api.types.is_numeric_dtype(df['Service_Date']):
                df['Date'] = pd.to_datetime(df['Service_Date'], unit='D', origin='1899-12-30')
            else:
                df['Date'] = pd.to_datetime(df['Service_Date'], errors='coerce')

            df['YearMonth'] = df['Date'].dt.to_period('M')
            stacked = (
                df.groupby(['YearMonth', 'Region'])['Billed_Amt']
                .sum().reset_index().sort_values('YearMonth')
            )
            stacked['Plot_Date'] = stacked['YearMonth'].dt.to_timestamp()

            fig = px.bar(
                stacked, x='Plot_Date', y='Billed_Amt', color='Region', barmode='stack',
                title=f"Baseline Monthly Billed Trend — {os.path.basename(uploaded_file_path)}",
                labels={'Billed_Amt': 'Total Billed ($)', 'Plot_Date': 'Month'}
            )
            fig.update_xaxes(dtick="M1", tickformat="%b %Y", tickangle=-30)
            display(go.FigureWidget(fig))

        except Exception as e:
            print(f"❌ Error reading file: {e}")
            print("Ensure your Excel file has sheets named 'fake enrollment' and 'fake claims'.")
            return

    with chat_area:
        clear_output(wait=True)
        print(f"📊 Baseline trend plotted for: {os.path.basename(uploaded_file_path)}\n")
        print("🤖 Agent ready. Try asking:")
        print("   • Which month had the biggest spike?")
        print("   • What drove the month-over-month change in paid claims?")
        print("   • Which region contributed most to the anomaly?")
        print("─" * 60)
    chat_box.layout.display = 'flex'

# ── Chat ─────────────────────────────────────────────────────
def on_chat_send(b=None):
    user_text = chat_input.value.strip()
    if not user_text:
        return
    chat_input.value = ''

    if not uploaded_file_path:
        with chat_area:
            print("⚠️  Please upload a claims file before asking questions.")
        return

    with chat_area:
        print(f"👤 You: {user_text}")

    full_prompt = (
        "You are a precise healthcare claims analytics assistant. "
        "Use your tools to investigate data files when asked. "
        "Respond concisely and in plain English.\n\n"
        f"[File: '{uploaded_file_path}']\nUser: {user_text}"
    )
    try:
        response    = agent.invoke({"messages": [("user", full_prompt)]})
        raw_content = response['messages'][-1].content
        agent_reply = (
            " ".join(item.get('text', '') for item in raw_content if isinstance(item, dict))
            if isinstance(raw_content, list) else str(raw_content)
        )
    except Exception as e:
        agent_reply = f"Error: {e}"

    with chat_area:
        print(f"🤖 Agent: {agent_reply.strip()}\n")
        print("─" * 60)
    with chart_area:
        clear_output(wait=True)

# ── Wire up & render ─────────────────────────────────────────
upload_btn.on_click(on_upload_click)
plot_btn.on_click(on_plot_click)
chat_btn.on_click(on_chat_send)
chat_input.on_submit(on_chat_send)

display(widgets.VBox([
    widgets.HTML("<h3 style='margin-bottom:4px'>🏥 Healthcare Claims AI Agent</h3>"),
    widgets.HTML("<p style='color:#555;margin-top:0'>Upload your Excel claims file to get started.</p>"),
    widgets.HBox([upload_btn, file_label], layout=widgets.Layout(align_items='center', gap='12px')),
    widgets.HBox([plot_btn]),
    chart_area,
    chat_area,
    chat_box
]))
